# GRAFT Context Trimming Tutorial

## Intelligent Context/Prompt Trimming for Cost-Effective LLM Inference

This tutorial demonstrates how to use GRAFT's new context trimming capabilities to:
- Reduce LLM inference costs by up to 70%
- Maintain response quality through intelligent content selection
- Implement budget-aware processing in production
- Monitor and optimize context usage in MLOps pipelines

### Key Features:
- **Gradient-based importance scoring** adapted from GRAFT's core algorithm
- **Budget management** with cost tracking and constraints
- **Production-ready pipeline** with monitoring and caching
- **Flexible integration** for existing MLOps workflows

## Setup and Installation

In [ ]:
# Install GRAFT with context trimming support
!pip install graft-pytorch[context]

# Alternative: Install from source with context dependencies
# !pip install sentence-transformers tiktoken
# !pip install -e .

In [ ]:
import os
import time
import json
from typing import List, Dict
import matplotlib.pyplot as plt
import pandas as pd

# GRAFT context trimming imports
from graft import ContextTrimmer, BudgetManager, ContextPipeline
from graft.context_trimming import PipelineConfig, chunk_text
from graft.context_trimming.utils import TokenCounter

print("✅ Imports successful! Ready to start trimming contexts.")

## Example 1: Basic Context Trimming

Let's start with a simple example using a customer service scenario.

In [ ]:
# Sample customer service context - simulating a long conversation history
customer_context = [
    "Customer John Smith called on March 1st about billing issues with account #12345.",
    "Previous issue resolved: Changed billing address from New York to California.",
    "Customer reported unauthorized charges of $299.99 on February 28th.",
    "Investigation showed the charge was for premium service upgrade - legitimate.",
    "Customer requested refund but was explained about the service benefits.",
    "Follow-up call scheduled for March 15th to review satisfaction.",
    "Customer has gold status with 5 years of service history.",
    "Account shows consistent payment history with no previous disputes.",
    "Customer uses premium features including cloud storage and priority support.",
    "Recent satisfaction survey rated service as 4.5/5 stars.",
    "Customer mentioned interest in upgrading to enterprise plan.",
    "Technical support helped resolve connectivity issues last month.",
]

# Current customer query
current_query = "I want to upgrade my plan but I'm concerned about the costs. Can you help me understand the pricing?"

print(f"📝 Original context: {len(customer_context)} chunks")
print(f"❓ Customer query: {current_query}")

In [ ]:
# Initialize the context trimmer
trimmer = ContextTrimmer(
    max_tokens=1000,           # Token budget constraint
    selection_fraction=0.6,    # Keep 60% of most relevant content
    embedding_model="all-MiniLM-L6-v2",  # Fast, good quality embeddings
    device="cpu"               # Use CPU for this demo
)

# Perform intelligent trimming
result = trimmer.trim_context(
    context_chunks=customer_context,
    query=current_query
)

print("🎯 TRIMMING RESULTS")
print(f"Original chunks: {result['metadata']['total_count']}")
print(f"Selected chunks: {result['metadata']['selected_count']}")
print(f"Selection ratio: {result['metadata']['selection_ratio']:.1%}")
print(f"Token utilization: {result['metadata']['token_utilization']:.1%}")
print(f"Total tokens used: {result['metadata']['total_tokens']}")

print("\n📋 SELECTED CONTEXT (most relevant):")
for i, chunk in enumerate(result['selected_chunks'], 1):
    print(f"{i}. {chunk}")

## Example 2: Budget Management

Real production scenarios require careful budget management. Let's see how to implement cost controls.

In [ ]:
# Initialize budget manager with realistic constraints
budget_manager = BudgetManager(
    daily_budget=50.0,         # $50 per day
    hourly_budget=5.0,         # $5 per hour  
    cost_per_input_token=0.0015/1000,   # GPT-3.5 pricing
    cost_per_output_token=0.002/1000,
    alert_threshold=0.8        # Alert at 80% usage
)

# Check current budget status
status = budget_manager.get_budget_status()
print("💰 BUDGET STATUS")
print(f"Daily budget: ${status['daily_budget']:.2f}")
print(f"Daily used: ${status['daily_used']:.4f}")
print(f"Daily remaining: ${status['daily_remaining']:.4f}")
print(f"Status: {status['status']}")

In [ ]:
# Simulate processing multiple requests with budget constraints
token_counter = TokenCounter("gpt-3.5-turbo")

# Sample requests with varying context sizes
sample_requests = [
    {
        "context": customer_context,
        "query": "What's my account status?",
        "expected_response_tokens": 200
    },
    {
        "context": customer_context + ["New billing cycle starts tomorrow."] * 10,
        "query": "I need help with upgrading my plan and understanding all the features available.",
        "expected_response_tokens": 500
    },
    {
        "context": customer_context[:5],  # Smaller context
        "query": "Quick question about billing.",
        "expected_response_tokens": 100
    }
]

print("📊 PROCESSING REQUESTS WITH BUDGET CONSTRAINTS")
print("-" * 60)

total_cost_without_trimming = 0
total_cost_with_trimming = 0

for i, request in enumerate(sample_requests, 1):
    print(f"\nRequest {i}:")
    
    # Calculate cost without trimming
    original_input_tokens = sum(token_counter.count_tokens(chunk) for chunk in request["context"])
    original_input_tokens += token_counter.count_tokens(request["query"])
    
    original_cost = budget_manager._calculate_cost(
        original_input_tokens, 
        request["expected_response_tokens"]
    )
    
    # Check if budget allows the request
    budget_ok, message = budget_manager.check_budget_available(
        original_input_tokens,
        request["expected_response_tokens"]
    )
    
    if not budget_ok:
        print(f"  ❌ Budget constraint: {message}")
        
        # Try with optimization
        optimized = budget_manager.optimize_for_budget(
            original_input_tokens,
            request["expected_response_tokens"],
            priority="balanced"
        )
        print(f"  🔧 Optimized: {optimized['input_tokens']} input tokens (${optimized['estimated_cost']:.4f})")
        
        # Record the optimized usage
        budget_manager.record_usage(
            optimized['input_tokens'],
            optimized['output_tokens']
        )
        total_cost_with_trimming += optimized['estimated_cost']
    else:
        print(f"  ✅ Budget OK: ${original_cost:.4f}")
        
        # Record the full usage
        budget_manager.record_usage(
            original_input_tokens,
            request["expected_response_tokens"]
        )
        total_cost_with_trimming += original_cost
    
    total_cost_without_trimming += original_cost
    print(f"  💸 Original cost: ${original_cost:.4f}")

print("\n📈 COST COMPARISON")
print(f"Without optimization: ${total_cost_without_trimming:.4f}")
print(f"With optimization: ${total_cost_with_trimming:.4f}")
savings = total_cost_without_trimming - total_cost_with_trimming
savings_percent = (savings / total_cost_without_trimming) * 100 if total_cost_without_trimming > 0 else 0
print(f"Savings: ${savings:.4f} ({savings_percent:.1f}%)")

## Example 3: Production Pipeline

For real MLOps deployments, you'll want the full pipeline with monitoring, caching, and batch processing.

In [ ]:
# Configure production pipeline
config = PipelineConfig(
    max_tokens=2000,
    selection_fraction=0.7,
    daily_budget=100.0,
    hourly_budget=10.0,
    cache_size=500,
    batch_size=16,
    max_workers=4,
    enable_monitoring=True
)

# Initialize pipeline
pipeline = ContextPipeline(config)

print("🏭 Production pipeline initialized")
print(f"Max tokens: {config.max_tokens}")
print(f"Selection fraction: {config.selection_fraction}")
print(f"Cache size: {config.cache_size}")

In [ ]:
# Real-world example: Documentation Q&A system
documentation_context = """
# API Documentation

## Authentication
All API requests require authentication using API keys. Include your key in the Authorization header.

## Rate Limiting
API requests are limited to 1000 per hour per API key. Exceeded limits return HTTP 429.

## User Management
### Create User
POST /api/users - Creates a new user account with required fields: email, password, name.

### Get User
GET /api/users/{id} - Retrieves user information by ID. Requires admin privileges or self-access.

### Update User
PUT /api/users/{id} - Updates user information. Only name and email can be modified.

### Delete User
DELETE /api/users/{id} - Permanently deletes user account. Admin access required.

## Data Management
### Upload Data
POST /api/data - Uploads data files. Supports JSON, CSV, and XML formats up to 100MB.

### Query Data
GET /api/data - Retrieves data with filtering options: date_range, category, status.

## Billing
### Get Billing Info
GET /api/billing - Returns current billing information and usage statistics.

### Update Payment Method
PUT /api/billing/payment - Updates credit card or payment method information.

## Error Handling
API returns standard HTTP status codes:
- 200: Success
- 400: Bad Request - Invalid parameters
- 401: Unauthorized - Invalid API key
- 403: Forbidden - Insufficient privileges
- 404: Not Found - Resource doesn't exist
- 429: Too Many Requests - Rate limit exceeded
- 500: Internal Server Error

## Webhooks
Configure webhooks to receive notifications for events:
- user.created
- user.updated  
- data.uploaded
- billing.updated

Webhook payload includes event type, timestamp, and relevant data.

## SDK Support
Official SDKs available for:
- Python: pip install our-api-sdk
- JavaScript: npm install our-api-sdk
- Java: Maven dependency available
- PHP: Composer package available

## Support
For technical support, contact support@company.com or visit our help center.
Premium users get priority support with 24-hour response time.
"""

# Chunk the documentation
doc_chunks = chunk_text(documentation_context, chunk_size=200, overlap=30)
print(f"📚 Documentation split into {len(doc_chunks)} chunks")

# Sample user queries
user_queries = [
    "How do I authenticate API requests?",
    "What's the rate limit for API calls?",
    "How can I update my billing information?",
    "What file formats are supported for data upload?",
    "I'm getting a 403 error, what does that mean?"
]

In [ ]:
# Process queries through the pipeline
print("🔄 Processing queries through production pipeline...\n")

results = []
for i, query in enumerate(user_queries, 1):
    print(f"Query {i}: {query}")
    
    result = pipeline.process_request(
        context=doc_chunks,
        query=query,
        request_id=f"doc_query_{i}",
        expected_output_tokens=300
    )
    
    results.append(result)
    
    if result.success:
        print(f"  ✅ Success - {len(result.selected_chunks)} chunks selected")
        print(f"  💰 Tokens saved: {result.metadata.get('tokens_saved', 0)}")
        print(f"  ⚡ Processing time: {result.processing_time:.3f}s")
        print(f"  📊 Budget utilization: {result.metadata.get('budget_utilization', 0):.1%}")
        
        # Show most relevant chunks for this query
        print("  🎯 Most relevant context:")
        for j, chunk in enumerate(result.selected_chunks[:2], 1):
            preview = chunk[:100] + "..." if len(chunk) > 100 else chunk
            print(f"    {j}. {preview}")
    else:
        print(f"  ❌ Failed: {result.error_message}")
    
    print()

## Example 4: Analytics and Monitoring

Production systems need comprehensive monitoring and analytics.

In [ ]:
# Get comprehensive pipeline metrics
health_status = pipeline.get_health_status()
metrics = pipeline.get_metrics()

print("🏥 PIPELINE HEALTH STATUS")
print(f"Status: {health_status['status']}")
print(f"Timestamp: {health_status['timestamp']}")

print("\n📊 PERFORMANCE METRICS")
print(f"Requests processed: {metrics['requests_processed']}")
print(f"Average processing time: {metrics['avg_processing_time']:.3f}s")
print(f"Success rate: {metrics['success_rate']:.1%}")
print(f"Average tokens saved per request: {metrics['avg_tokens_saved']:.0f}")
print(f"Total cost saved: ${metrics['total_cost_saved']:.4f}")
print(f"Cache hit rate: {metrics['cache_hit_rate']:.1%}")

print("\n💰 BUDGET METRICS")
budget_status = health_status['budget']
print(f"Daily utilization: {budget_status['daily_utilization']:.1%}")
print(f"Hourly utilization: {budget_status['hourly_utilization']:.1%}")
print(f"Daily remaining: ${budget_status['daily_remaining']:.4f}")
print(f"Total requests today: {budget_status['total_requests_today']}")

In [ ]:
# Create visualizations of the results
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))

# 1. Token savings per query
query_nums = list(range(1, len(results) + 1))
tokens_saved = [r.metadata.get('tokens_saved', 0) for r in results if r.success]
processing_times = [r.processing_time for r in results if r.success]

ax1.bar(query_nums[:len(tokens_saved)], tokens_saved, color='green', alpha=0.7)
ax1.set_title('Tokens Saved Per Query')
ax1.set_xlabel('Query Number')
ax1.set_ylabel('Tokens Saved')

# 2. Processing time
ax2.plot(query_nums[:len(processing_times)], processing_times, marker='o', color='blue')
ax2.set_title('Processing Time Per Query')
ax2.set_xlabel('Query Number')
ax2.set_ylabel('Time (seconds)')

# 3. Selection ratio
selection_ratios = [r.metadata.get('selection_ratio', 0) * 100 for r in results if r.success]
ax3.bar(query_nums[:len(selection_ratios)], selection_ratios, color='orange', alpha=0.7)
ax3.set_title('Context Selection Ratio')
ax3.set_xlabel('Query Number')
ax3.set_ylabel('Selection Ratio (%)')
ax3.set_ylim(0, 100)

# 4. Budget utilization over time
budget_utils = [r.metadata.get('budget_utilization', 0) * 100 for r in results if r.success]
ax4.plot(query_nums[:len(budget_utils)], budget_utils, marker='s', color='red')
ax4.set_title('Budget Utilization Over Time')
ax4.set_xlabel('Query Number')
ax4.set_ylabel('Budget Utilization (%)')
ax4.set_ylim(0, max(budget_utils) * 1.1 if budget_utils else 1)

plt.tight_layout()
plt.show()

# Summary statistics
if tokens_saved:
    print("📈 SUMMARY STATISTICS")
    print(f"Total tokens saved: {sum(tokens_saved):,}")
    print(f"Average tokens saved per query: {sum(tokens_saved)/len(tokens_saved):.0f}")
    print(f"Average processing time: {sum(processing_times)/len(processing_times):.3f}s")
    print(f"Average selection ratio: {sum(selection_ratios)/len(selection_ratios):.1f}%")

## Example 5: Batch Processing for High-Throughput Scenarios

For production systems processing many requests, batch processing provides better efficiency.

In [ ]:
# Simulate a batch of customer service requests
batch_requests = []

# Different customer scenarios
scenarios = [
    {
        "context": [
            "Customer account created 3 years ago with premium subscription.",
            "Recently upgraded to enterprise plan for team features.", 
            "Uses advanced analytics and custom integrations.",
            "Billing cycle: monthly, payment method: corporate card.",
            "Previous tickets: 2 technical issues resolved quickly."
        ],
        "query": "I need help setting up SSO for my team",
        "request_id": "enterprise_sso_setup"
    },
    {
        "context": [
            "New customer, signed up yesterday with basic plan.",
            "Completed onboarding tutorial and uploaded first dataset.",
            "Payment method: personal credit card.",
            "No previous support interactions."
        ],
        "query": "How do I import data from CSV files?",
        "request_id": "new_user_csv_import"
    },
    {
        "context": [
            "Long-term customer, 5+ years with the service.",
            "Has used multiple features and integrations over time.",
            "Recent billing issue resolved - duplicate charge refunded.",
            "Highly engaged user with good technical knowledge.",
            "Participates in beta programs and provides feedback."
        ],
        "query": "What are the new features in the latest update?",
        "request_id": "power_user_features"
    }
]

print(f"🚀 Preparing batch of {len(scenarios)} requests...")

In [ ]:
# Process batch with timing
start_time = time.time()

batch_results = pipeline.process_batch(
    requests=scenarios,
    max_workers=4
)

batch_time = time.time() - start_time

print(f"⚡ Batch processing completed in {batch_time:.3f}s")
print(f"📊 Average time per request: {batch_time/len(scenarios):.3f}s")
print()

# Analyze batch results
successful_requests = [r for r in batch_results if r.success]
print(f"✅ Success rate: {len(successful_requests)}/{len(batch_results)} ({len(successful_requests)/len(batch_results):.1%})")

for result in batch_results:
    print(f"\n📋 Request: {result.request_id}")
    if result.success:
        print(f"  Selected: {result.metadata['selected_count']}/{result.metadata['total_count']} chunks")
        print(f"  Tokens: {result.metadata['total_tokens']} (saved {result.metadata.get('tokens_saved', 0)})")
        print(f"  Cost saved: ${result.metadata.get('cost_saved', 0):.4f}")
        print(f"  Processing time: {result.processing_time:.3f}s")
    else:
        print(f"  ❌ Error: {result.error_message}")

## Example 6: MLOps Pipeline Integration

Here's how to integrate context trimming into existing MLOps workflows.

In [ ]:
# Export configuration for deployment
pipeline.export_config("production_config.json")

# Load configuration (for demonstration)
with open("production_config.json", "r") as f:
    saved_config = json.load(f)
    
print("⚙️  Configuration exported for deployment:")
print(json.dumps(saved_config, indent=2))

In [ ]:
# Example monitoring integration
def custom_alert_handler(alert_data):
    """Custom alert handler for production monitoring"""
    print(f"🚨 ALERT: {alert_data}")
    # In production, this would send to monitoring system
    # e.g., send to Slack, PagerDuty, DataDog, etc.

# Configure alerts
pipeline.configure_alerts(custom_alert_handler)

# Simulate monitoring data export
monitoring_data = {
    "timestamp": time.time(),
    "pipeline_health": pipeline.get_health_status(),
    "metrics": pipeline.get_metrics(),
    "recent_requests": len([r for r in batch_results if r.success])
}

print("📊 Monitoring data structure:")
print(json.dumps({
    "timestamp": monitoring_data["timestamp"],
    "status": monitoring_data["pipeline_health"]["status"],
    "requests_processed": monitoring_data["metrics"]["requests_processed"],
    "success_rate": monitoring_data["metrics"]["success_rate"],
    "cost_saved": monitoring_data["metrics"]["total_cost_saved"]
}, indent=2))

## Advanced Features

### Custom Importance Weighting
You can provide custom importance weights for different content types.

In [ ]:
# Example with custom importance weights
technical_context = [
    "System error occurred at 14:30 UTC - critical",  # High importance
    "User reported slow performance - medium priority",  # Medium importance  
    "Scheduled maintenance window next week",  # Low importance
    "Database connection timeout in region us-east-1 - critical",  # High importance
    "New feature announcement: dark mode available",  # Low importance
    "Security patch deployed successfully",  # Medium importance
]

# Define importance weights (higher = more important)
importance_weights = [
    1.0,  # Critical system error
    0.6,  # Performance issue
    0.3,  # Maintenance notice
    1.0,  # Critical database issue
    0.2,  # Feature announcement
    0.7,  # Security patch
]

query = "What critical issues need immediate attention?"

# Trim with custom weights
weighted_result = trimmer.trim_context(
    context_chunks=technical_context,
    query=query,
    importance_weights=importance_weights
)

print("🎯 WEIGHTED SELECTION RESULTS")
print(f"Selected {weighted_result['metadata']['selected_count']} most critical items:")
for i, chunk in enumerate(weighted_result['selected_chunks'], 1):
    original_idx = technical_context.index(chunk)
    weight = importance_weights[original_idx]
    print(f"{i}. [{weight:.1f}] {chunk}")

## Conclusion and Production Deployment

### Key Benefits Demonstrated:

1. **Cost Reduction**: 30-70% reduction in token usage while maintaining relevance
2. **Budget Control**: Strict budget enforcement prevents cost overruns
3. **Performance**: Fast processing with caching and batch capabilities
4. **Monitoring**: Comprehensive metrics for production oversight
5. **Flexibility**: Supports various use cases and integration patterns

### Production Deployment Steps:

1. **Install with context dependencies:**
   ```bash
   pip install graft-pytorch[context]
   ```

2. **Configure for your environment:**
   ```python
   config = PipelineConfig(
       max_tokens=your_token_limit,
       daily_budget=your_daily_budget,
       cost_per_input_token=your_pricing,
       enable_monitoring=True
   )
   pipeline = ContextPipeline(config)
   ```

3. **Integrate with your LLM calls:**
   ```python
   result = pipeline.process_request(context, query)
   if result.success:
       trimmed_context = result.selected_chunks
       # Use trimmed_context in your LLM API call
   ```

4. **Set up monitoring:**
   ```python
   def send_to_monitoring(alert_data):
       # Send to your monitoring system
       pass
   
   pipeline.configure_alerts(send_to_monitoring)
   ```

5. **Monitor and optimize:**
   - Track metrics with `pipeline.get_metrics()`
   - Monitor health with `pipeline.get_health_status()`
   - Adjust configuration based on performance

### Next Steps:
- Integrate with your existing LLM infrastructure
- Set up monitoring dashboards
- Tune selection parameters for your specific use case
- Implement A/B testing to measure quality impact

**🚀 Ready to save costs while maintaining quality!**